In [ ]:
# === Setup ===
# Runtime: <3m with OAI_FAST_MODE=1
# Hardware: CPU smoke; GPU recommended for full run
# Network: optional
# Competition-safe: No — learning profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# Competition starter — Image Classification

Pipeline chạy được bằng nearest-centroid baseline. TODO: thay baseline bằng CNN, giữ split/metric/submission contract.

In [ ]:
def make_images(n=120,seed=42):
    rng=np.random.default_rng(seed); X=np.zeros((n,1,16,16),np.float32); y=np.arange(n)%2
    for i,label in enumerate(y):
        if label==0: X[i,0,3:13,6:10]=1.0
        else: X[i,0,6:10,3:13]=1.0
        X[i]+=rng.normal(0,.08,X[i].shape)
    order=rng.permutation(n); return X[order],y[order]
X,y=make_images(80 if FAST_MODE else 240); cut=int(.75*len(X)); Xtr,Xva=X[:cut],X[cut:]; ytr,yva=y[:cut],y[cut:]
print("train/val",Xtr.shape,Xva.shape,"balance",np.bincount(ytr))

In [ ]:
# EDA + preprocess
assert Xtr.dtype==np.float32 and set(np.unique(ytr))=={0,1}
means=np.stack([Xtr[ytr==c].mean(0) for c in (0,1)])
distance=((Xva[:,None]-means[None])**2).mean((2,3,4)); pred=distance.argmin(1); accuracy=(pred==yva).mean()
assert accuracy>.9
submission=np.c_[np.arange(len(pred)),pred]
print("validation accuracy",accuracy,"submission shape",submission.shape)
# TODO: train a small CNN and compare using exactly the same validation indices.